In [ ]:
from pathlib import Path

from astropy.table import unique, vstack

%matplotlib inline
import matplotlib.pyplot as plt

from stellarphot.utils.magnitude_transforms import transform_to_catalog

from stellarphot import PhotometryData

## File names

In [ ]:
input_file_name = "photometry.ecsv"

In [ ]:
input_photometry_file = Path(input_file_name)
output_photometry_file = input_photometry_file.parent / (input_photometry_file.stem + "-transformed" + input_photometry_file.suffix)
output_photometry_file

# Parameters

The magnitudes in each image are fit using this model:

$$
r_{p, c} = r_{p, inst} + a r_{p, inst} + b r_{p, inst}^2 + c (m_1 - m_2)_c + d (m_1 - m_2)_c^2 + z
$$

where $(m_1 - m_2)_c$ is a catalog color. Which color that is depends on the band being calibrated -- $B - V$ for $B$ and $V$, $R - I$ for $R$ and $I$, and so on. `transform_to_catalog` picks the conventional pair for the band unless you pass `cat_color` yourself.

The leading $r_{p, inst}$ is there because `fit_diff` defaults to `True`: what is fit is the catalog magnitude *minus* the instrumental one, and the instrumental magnitude is added back afterwards. That makes $a$ a small correction to the slope rather than the slope itself -- it comes out near zero, not near one.

`vary` in the cell below lists which of those terms are fit. Every term not listed is held at exactly zero, which is how you fix a parameter.

`expected` gives the range each fitted term is expected to land in. This is a check on the answer, not a constraint on the fit: a term that comes out outside its range is reported in a warning, with the value it actually reached, rather than being clamped to the edge of the range.

### *Recommendation:*

+ Start with $a$, $c$ and $z$ in `vary`, leaving the quadratic terms $b$ and $d$ out so they stay at zero. That is where this workflow has always started.
+ If the fit with $b$ and $d$ fixed is not good enough -- large residuals, or structure left in a plot of catalog magnitude minus calibrated magnitude against instrumental magnitude or color -- add $b$ and/or $d$ to `vary` and see whether the fit improves.


In [ ]:
# Terms of the transform to fit. The others are held at zero. The terms are
# "a" and "b" for the linear and quadratic dependence on instrumental
# magnitude, "c" and "d" for the same in color, and "z" for the zero point.
vary = ("a", "c", "z")

# Range each fitted term is expected to land in. This does not constrain the
# fit -- a value outside its range is reported in a warning, not clamped.
expected = {"z": (12, 25)}

catalog = "apass_dr9"   # or "refcat2"

In [ ]:
all_mags = PhotometryData.read(input_photometry_file)
all_mags.add_bjd_col()

## Do the transforms, one filter at a time

In [ ]:
output_table = []
filter_groups = all_mags.group_by("passband")

for key, group in zip(filter_groups.groups.keys, filter_groups.groups):
    # The key is a table column, not a value...
    k = key[0]
    print(f"Transforming band {k}")
    by_bjd = group.group_by("file")

    # The catalog band and color are not passed: each defaults to the band
    # being observed and to that band's conventional color pair.
    transform_to_catalog(
        by_bjd,
        k,
        obs_error_column="mag_error",
        vary=vary,
        expected=expected,
        cat_name=catalog,
        in_place=True,
    )
    output_table.append(by_bjd.copy())
    one_image = by_bjd[by_bjd["file"] == list(set(by_bjd["file"]))[0]]
    plt.figure()
    plt.plot(one_image["mag_inst"], one_image["mag_cat"], ".", alpha=0.4)
    #plt.plot([-10.665, -4.703], [9.59, 15.339])
    plt.xlabel(f"Instrumental magnitude {k}")
    plt.ylabel(f"Catalog mag {k}")
    plt.grid()
output_table = vstack(output_table, join_type="outer")

## How well did each image fit?

`z_error` is the uncertainty of the zero point, `fit_redchi` describes the fit as a whole, and three further columns describe how the fit was *weighted*. All of them are properties of an image rather than of a star, so they are repeated down every row of that image and one row per image is the whole story.

Read `fit_redchi` first. With `obs_error_column` given it is a reduced chi-square: **near 1** means the scatter of the stars about the transform is about what `mag_error` says it should be, **much larger than 1** means the stars scatter more than their errors admit -- a bad night, a bad field, or errors that are too small. It is worth reading before trusting `mag_cal_error`, because that error believes the errors as quoted: an image whose `fit_redchi` is far from one is reporting a `mag_cal_error` that is off by roughly the square root of that factor.

Do not stop there. `fit_redchi` is a ratio, so a fit whose errors are wrong in the right way reports a healthy-looking one -- a single star can quietly hold most of a fit's weight (see issue #694). The other three columns are what catch that.

- `fit_cat_error_missing_frac` -- the fraction of the fitted stars whose *catalog* error could not be used. APASS DR9 reports an error of exactly zero for most of its B stars but almost none of its V stars, so this can be large in one band and tiny in another for the same field. Where it is large, that band's `fit_redchi` cannot be compared with another band's.
- `fit_max_weight_share` -- the largest share of the fit's weight held by any one star. A fit spread evenly over N stars reads `1/N`; anything approaching 1 is one star running the fit.
- `fit_excess_scatter` -- the scatter, in magnitudes, that would have to be added to every star's uncertainty to bring `fit_redchi` to 1. Flat-field gradients across the field and the catalog's own photometry both land here, and no weighting scheme can repair either.

An image with a `fit_redchi` far from the others, a `z_error` much larger than the others, or a `fit_max_weight_share` far above `1/N` is the one to look at.

In [ ]:
fit_quality = unique(
    output_table[
        "passband",
        "file",
        "z",
        "z_error",
        "fit_redchi",
        "fit_cat_error_missing_frac",
        "fit_max_weight_share",
        "fit_excess_scatter",
    ],
    keys=["passband", "file"],
)
fit_quality.pprint_all()

In [ ]:
one_image = by_bjd[by_bjd["file"] == list(set(by_bjd["file"]))[0]]

In [ ]:
plt.figure()
plt.plot(one_image["mag_inst"], one_image["mag_cat"], ".", alpha=0.4)
#plt.plot([-10.665, -4.703], [9.59, 15.339])
plt.xlabel("Instrumental magnitude")
plt.ylabel("Catalog mag")
plt.grid()

In [ ]:
plt.figure()
plt.plot(one_image["snr"], one_image["mag_cat"] - one_image["mag_cal"], ".", alpha=0.4)
plt.xlabel("SNR")
plt.ylabel("Catalog mag - feder mag")
plt.grid()

In [ ]:
plt.figure()
plt.plot(one_image["mag_cal"], one_image["mag_cat"] - one_image["mag_cal"], ".", alpha=0.4)
plt.xlabel("calibrated Feder mag")
plt.ylabel("Catalog mag - feder mag")
plt.grid()

In [ ]:
plt.figure()
plt.plot(one_image["mag_inst"], one_image["mag_cat"] - one_image["mag_cal"], ".", alpha=0.4)
plt.xlabel("Instrumental Feder mag")
plt.ylabel("Catalog mag - feder mag")
plt.grid()

In [ ]:
plt.figure()
plt.plot(one_image["color_cat"], one_image["mag_cat"] - one_image["mag_cal"], ".", alpha=0.4)
plt.xlabel("catalog color")
plt.ylabel("Catalog mag - feder mag")
plt.grid()

In [ ]:
rp_only = output_table[output_table["passband"] == "SR"]

fig, (ax_z, ax_chi) = plt.subplots(2, 1, sharex=True, figsize=(8, 6))

# The error bars are the fit's own uncertainty in the zero point, so a night
# whose z wanders by much more than they allow is telling you something real.
ax_z.errorbar(
    rp_only["bjd"].value, rp_only["z"], yerr=rp_only["z_error"], fmt=".", alpha=0.4
)
ax_z.set_ylabel("z")
ax_z.grid()

ax_chi.plot(rp_only["bjd"].value, rp_only["fit_redchi"], ".", alpha=0.4)
ax_chi.axhline(1, color="gray", linestyle=":")
ax_chi.set_ylabel("fit_redchi")
ax_chi.set_xlabel("BJD")
ax_chi.grid()

In [ ]:
output_table.write(output_photometry_file, overwrite=True)